In [ ]:
!pip install streamlit

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-chroma langchain-openai chromadb datasets sentence-transformers fastapi uvicorn python-dotenv bitsandbytes accelerate peft deep_translator

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 106.1 MB/s eta 0:00:

In [ ]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 3s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠧

In [ ]:
%%writefile train.py
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from datasets import load_dataset
import os
os.environ["WANDB_MODE"] = "disabled"
# 설정 (빠른 확인을 위해 데이터 2000개, 1 Epoch만 학습)
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_PATH = "./models/movie_embedding_finetuned"
DATASET_NAME = "stzhao/movie_posters_100k_controlnet"
BATCH_SIZE = 32
EPOCHS = 1
TRAIN_LIMIT = 2000

def main():
    print(f"🚀 임베딩 모델 학습 시작! (Base: {MODEL_NAME})")
    model = SentenceTransformer(MODEL_NAME)

    print(f"📥 데이터셋 로드 중 (상위 {TRAIN_LIMIT}개)...")
    dataset = load_dataset(DATASET_NAME, split="train", streaming=True)

    train_examples = []
    for i, item in enumerate(dataset):
        if i >= TRAIN_LIMIT: break

        title = item.get('title', '')
        genres = item.get('genres', '')
        content = item.get('text', item.get('caption', ''))

        if not title or not content: continue

        query_text = f"Movie title: {title}, Genre: {genres}"
        train_examples.append(InputExample(texts=[query_text, content]))

    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=EPOCHS,
        warmup_steps=100,
        output_path=OUTPUT_PATH,
        show_progress_bar=True
    )
    print(f"✅ 학습 완료! 저장 경로: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

Overwriting train.py


In [ ]:
%%writefile ingestion.py
import os
import shutil
from datasets import load_dataset
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 경로 설정 (train.py 결과물 사용)
DATASET_NAME = "stzhao/movie_posters_100k_controlnet"
DB_PATH = "./chroma_db_final"
MODEL_PATH = "./models/movie_embedding_finetuned" # 학습된 모델
INGEST_LIMIT = 2000 # DB 저장 개수

def main():
    if os.path.exists(DB_PATH):
        shutil.rmtree(DB_PATH)

    print(f"📥 데이터셋 로드 및 DB 구축 시작 (모델: {MODEL_PATH})...")

    # 학습된 모델이 없으면 기본 모델 사용 (안전장치)
    if os.path.exists(MODEL_PATH):
        emb_model = MODEL_PATH
    else:
        emb_model = "sentence-transformers/all-MiniLM-L6-v2"
        print("⚠️ 학습된 모델이 없어 기본 모델을 사용합니다.")

    embedding_model = HuggingFaceEmbeddings(model_name=emb_model, model_kwargs={'device': 'cuda'})
    vectorstore = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
    dataset = load_dataset(DATASET_NAME, split="train", streaming=True)

    docs = []
    for i, item in enumerate(dataset):
        if i >= INGEST_LIMIT: break

        content = item.get("text", item.get("caption", ""))
        title = item.get("title", "Unknown")
        if not content: continue

        doc = Document(
            page_content=content,
            metadata={"id": i, "title": title}
        )
        docs.append(doc)

        if len(docs) >= 500:
            vectorstore.add_documents(docs)
            docs = []
            print(f"   💾 {i+1}개 처리 중...")

    if docs:
        vectorstore.add_documents(docs)

    print("✅ DB 구축 완료!")

if __name__ == "__main__":
    main()

Writing ingestion.py


In [ ]:
%%writefile app.py
import streamlit as st
import torch
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from deep_translator import GoogleTranslator
from datasets import load_dataset

# ✅ 경로 설정
DB_PATH = "./chroma_db_final"
MODEL_PATH = "./models/movie_embedding_finetuned"
LLM_ID = "NousResearch/Meta-Llama-3-8B-Instruct"
DATASET_NAME = "stzhao/movie_posters_100k_controlnet"

st.set_page_config(page_title="Movie Chatbot", layout="wide")
st.title("🎬 Movie Poster AI")

@st.cache_resource
def load_resources():
    print("DEBUG: 리소스 로딩 시작")

    # 1. 임베딩 모델 로드
    if os.path.exists(MODEL_PATH):
        emb_path = MODEL_PATH
    else:
        emb_path = "sentence-transformers/all-MiniLM-L6-v2"

    embedding_model = HuggingFaceEmbeddings(
        model_name=emb_path,
        model_kwargs={'device': 'cuda'}
    )
    vectorstore = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)

    # 2. LLM 로드
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    llm = AutoModelForCausalLM.from_pretrained(
        LLM_ID,
        quantization_config=bnb_config,
        device_map="cuda:0"
    )
    tokenizer = AutoTokenizer.from_pretrained(LLM_ID)

    # ⭐️ [수정된 부분] return_full_text=False 추가!
    # 이것을 추가해야 프롬프트를 제외하고 "답변"만 깔끔하게 나옵니다.
    pipe = pipeline(
        "text-generation",
        model=llm,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.7,
        return_full_text=False
    )

    # 3. 데이터셋 로드 (RAM 절약)
    ds = load_dataset(DATASET_NAME, split="train", streaming=True)
    ds_cache = []
    for i, item in enumerate(ds):
        if i >= 50: break
        ds_cache.append(item)

    return vectorstore, pipe, tokenizer, ds_cache

# 로딩 표시
with st.spinner("AI 모델 로딩 중..."):
    vectorstore, generator, tokenizer, ds_cache = load_resources()

if "messages" not in st.session_state: st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])
        if "images" in msg:
            cols = st.columns(len(msg["images"]))
            for i, img in enumerate(msg["images"]):
                cols[i].image(img["image"], caption=img["title"])

if prompt := st.chat_input("질문하세요 (예: 액션 영화 추천해줘)"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"): st.write(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                en_query = GoogleTranslator(source='auto', target='en').translate(prompt)
            except:
                en_query = prompt

            docs = vectorstore.similarity_search(en_query, k=3)
            context, found_imgs = "", []

            for doc in docs:
                title = doc.metadata.get("title", "")
                idx = doc.metadata.get("id")
                context += f"Title: {title}\nDesc: {doc.page_content}\n\n"

                if idx is not None and idx < len(ds_cache):
                    found_imgs.append({"image": ds_cache[int(idx)]['image'], "title": title})

            # Llama-3 프롬프트 템플릿 적용
            prompt_tmpl = [
                {"role": "system", "content": "You are a movie expert. Answer in Korean based on context."},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{prompt}"}
            ]
            final_prompt = tokenizer.apply_chat_template(prompt_tmpl, tokenize=False, add_generation_prompt=True)

            # 생성 실행
            res = generator(final_prompt)[0]['generated_text']

            # 후처리: 혹시라도 특수 토큰이 남아있으면 제거
            res = res.replace("<|eot_id|>", "").replace("<|start_header_id|>", "").replace("<|end_header_id|>", "").strip()

            # 한국어 번역 후처리
            try:
                if not any('가' <= c <= '힣' for c in res):
                    res = GoogleTranslator(source='auto', target='ko').translate(res)
            except: pass

            st.write(res)
            if found_imgs:
                cols = st.columns(len(found_imgs))
                for i, img in enumerate(found_imgs):
                    cols[i].image(img["image"], caption=img["title"])

            st.session_state.messages.append({"role": "assistant", "content": res, "images": found_imgs})

Overwriting app.py


In [ ]:
!python train.py

2025-11-26 20:48:17.951520: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764190098.057067    4039 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764190098.097146    4039 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764190098.204463    4039 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764190098.204519    4039 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764190098.204529    4039 computation_placer.cc:177] computation placer alr

In [ ]:
!python ingestion.py

📥 데이터셋 로드 및 DB 구축 시작 (모델: ./models/movie_embedding_finetuned)...
2025-11-26 20:49:51.974724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764190191.995334    4513 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764190192.001336    4513 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764190192.016323    4513 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764190192.016348    4513 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764190192

In [ ]:
!pkill -9 streamlit
!pkill -9 cloudflared

In [ ]:
!python inference.py

2025-11-26 21:50:24.893703: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764193824.913845   20979 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764193824.920037   20979 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764193824.935500   20979 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764193824.935531   20979 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764193824.935535   20979 computation_placer.cc:177] computation placer alr

In [ ]:
import time

# 1. 기존 프로세스 정리 (충돌 방지)
print("🧹 기존 프로세스 정리 중...")
!pkill -9 streamlit
!pkill -9 cloudflared

# 2. Streamlit 백그라운드 실행
print("🚀 Streamlit 서버 시작...")
!streamlit run app.py &>/content/logs.txt &

# 3. Cloudflare Tunnel 다운로드 및 실행
print("☁️ Cloudflare Tunnel 연결 중...")
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
# 로그를 tunnel_log.txt에 저장해서 URL을 추출합니다.
!./cloudflared tunnel --url http://localhost:8501 > /content/tunnel_log.txt 2>&1 &

# 4. URL 추출 및 출력
time.sleep(5) # 터널이 열릴 때까지 잠시 대기
print("🔗 아래 링크를 클릭하세요! (trycloudflare.com 링크)")
!grep -o 'https://.*\.trycloudflare.com' /content/tunnel_log.txt | head -n 1

🧹 기존 프로세스 정리 중...
🚀 Streamlit 서버 시작...
☁️ Cloudflare Tunnel 연결 중...
🔗 아래 링크를 클릭하세요! (trycloudflare.com 링크)
https://asked-convinced-scsi-watch.trycloudflare.com
